# Exercise for Graph V: Automatic Higher or Lower Game (Looping Graph)

### Your Task:
Implement an **Automatic Higher or Lower Game** using LangGraph loops and conditional edges!

- **Bounds:** Between `1` and `20` (`lower_bound = 1`, `upper_bound = 20`).
- **Guessing Logic:** The graph keeps guessing (up to a max of `7` guesses).
- **Hint Node:** Each time a number is guessed, the hint node tells whether the target is `"higher"` or `"lower"`, and adjusts bounds accordingly.
- **Stopping Condition:** 
  - If the guess is correct (`guess == target_number`), stop the game.
  - If not correct, keep looping until the maximum limit of `7` guesses is reached.
- **Input Format:**
  ```json
  {
    "player_name": "Student",
    "guesses": [],
    "attempts": 0,
    "lower_bound": 1,
    "upper_bound": 20
  }
  ```
- **Hint:** It will need to adjust its bounds after every guess based on the hint provided by the hint node.

In [1]:
# Install dependencies if needed:
# !pip install langgraph langchain-core

## 1. Define the State

We define `GameState` with:
- `player_name`: name of the player
- `target_number`: secret target number to be guessed (default or provided)
- `lower_bound`, `upper_bound`: current search range (initial 1 to 20)
- `current_guess`: the latest guess generated
- `guesses`: history list of all guesses made
- `attempts`: number of guesses made so far (max 7)
- `hint`: feedback from the hint node (`'higher'`, `'lower'`, or `'correct'`)
- `game_over`: boolean flag indicating whether the game should stop
- `result`: final message

In [2]:
from typing import TypedDict, List, Optional
import random
from langgraph.graph import StateGraph, START, END

class GameState(TypedDict, total=False):
    player_name: str
    target_number: int
    lower_bound: int
    upper_bound: int
    current_guess: int
    guesses: List[int]
    attempts: int
    hint: str
    game_over: bool
    result: str

## 2. Define the Nodes and Conditional Routing

1. **`guess_node`**: Guesses the midpoint `(lower_bound + upper_bound) // 2` (binary search strategy), increments attempts, and appends to the `guesses` list.
2. **`hint_node`**: Compares `current_guess` against `target_number`:
   - If equal $\rightarrow$ `hint = "correct"`, `game_over = True`
   - If `attempts >= 7` $\rightarrow$ `game_over = True` (max attempts reached)
   - If guess < target $\rightarrow$ `hint = "higher"`, `lower_bound = guess + 1`
   - If guess > target $\rightarrow$ `hint = "lower"`, `upper_bound = guess - 1`
3. **`check_loop_condition`**: If `game_over` is True, returns `END`; otherwise, loops back to `"guess_node"`.

In [3]:
# Node 1: Guessing Node
def guess_node(state: GameState) -> dict:
    low = state.get("lower_bound", 1)
    high = state.get("upper_bound", 20)
    
    # Optimal guessing strategy (binary search midpoint)
    guess = (low + high) // 2
    
    attempts = state.get("attempts", 0) + 1
    guesses = list(state.get("guesses", [])) + [guess]
    
    print(f"[Attempt {attempts}] Guessing: {guess} (Current Search Range: [{low}, {high}])")
    
    return {
        "current_guess": guess,
        "attempts": attempts,
        "guesses": guesses
    }

# Node 2: Hint & Bounds Adjustment Node
def hint_node(state: GameState) -> dict:
    guess = state["current_guess"]
    target = state.get("target_number", 14)  # default target if not passed
    attempts = state["attempts"]
    player = state.get("player_name", "Player")
    low = state["lower_bound"]
    high = state["upper_bound"]
    
    if guess == target:
        hint = "correct"
        result = f"Congratulations {player}! You guessed the target {target} correctly in {attempts} attempt(s)!"
        game_over = True
        print(f" -> Hint: Correct! Target {target} found!")
    elif attempts >= 7:
        hint = "higher" if guess < target else "lower"
        result = f"Game Over for {player}! Reached maximum limit of 7 guesses. The target was {target}."
        game_over = True
        print(f" -> Max attempts (7) reached. Game over!")
    elif guess < target:
        hint = "higher"
        low = guess + 1
        result = f"Guess {guess} is too low. Hint: Go higher!"
        game_over = False
        print(f" -> Hint: Higher! Adjusted bounds to [{low}, {high}]")
    else:
        hint = "lower"
        high = guess - 1
        result = f"Guess {guess} is too high. Hint: Go lower!"
        game_over = False
        print(f" -> Hint: Lower! Adjusted bounds to [{low}, {high}]")
        
    return {
        "hint": hint,
        "lower_bound": low,
        "upper_bound": high,
        "result": result,
        "game_over": game_over
    }

# Conditional Edge: Decides whether to continue looping or stop
def check_loop_condition(state: GameState):
    if state.get("game_over", False):
        return END
    return "guess_node"

## 3. Construct and Compile the Looping Graph

In [4]:
builder = StateGraph(GameState)

# Add nodes
builder.add_node("guess_node", guess_node)
builder.add_node("hint_node", hint_node)

# Define workflow
builder.add_edge(START, "guess_node")
builder.add_edge("guess_node", "hint_node")

# Looping conditional edge
builder.add_conditional_edges(
    "hint_node",
    check_loop_condition,
    {
        "guess_node": "guess_node",
        END: END
    }
)

game_graph = builder.compile()

## 4. Run the Game with Target Number 14

Input state matching the task specifications.

In [5]:
initial_input = {
    "player_name": "Student",
    "target_number": 14,
    "guesses": [],
    "attempts": 0,
    "lower_bound": 1,
    "upper_bound": 20
}

final_state = game_graph.invoke(initial_input)

print("\n=== FINAL STATE ===")
print(f"Player: {final_state['player_name']}")
print(f"Total Attempts: {final_state['attempts']}")
print(f"Guesses Made: {final_state['guesses']}")
print(f"Final Result: {final_state['result']}")

[Attempt 1] Guessing: 10 (Current Search Range: [1, 20])
 -> Hint: Higher! Adjusted bounds to [11, 20]
[Attempt 2] Guessing: 15 (Current Search Range: [11, 20])
 -> Hint: Lower! Adjusted bounds to [11, 14]
[Attempt 3] Guessing: 12 (Current Search Range: [11, 14])
 -> Hint: Higher! Adjusted bounds to [13, 14]
[Attempt 4] Guessing: 13 (Current Search Range: [13, 14])
 -> Hint: Higher! Adjusted bounds to [14, 14]
[Attempt 5] Guessing: 14 (Current Search Range: [14, 14])
 -> Hint: Correct! Target 14 found!

=== FINAL STATE ===
Player: Student
Total Attempts: 5
Guesses Made: [10, 15, 12, 13, 14]
Final Result: Congratulations Student! You guessed the target 14 correctly in 5 attempt(s)!


## 5. Test with a Random Target Number (1 to 20)

In [6]:
random_target = random.randint(1, 20)
print(f"Testing with Random Secret Target: {random_target}\n")

random_input = {
    "player_name": "Student",
    "target_number": random_target,
    "guesses": [],
    "attempts": 0,
    "lower_bound": 1,
    "upper_bound": 20
}

random_final = game_graph.invoke(random_input)
print(f"\nSummary: {random_final['result']}")
print(f"Guesses: {random_final['guesses']}")